# 15. Post-processing of the ensemble masks

Applies boundary-refinement methods to the Dice-weighted ensemble maps
(notebook 13) and evaluates each variant on the test split:

1. Morphology and image-guided smoothing.
2. Test-time augmentation of the averaged probability maps
   (plain, + bilateral filter, + edge-preserving filter).
3. Colour- and region-based refinement (GrabCut, SLIC superpixels,
   region growing, adaptive colour, texture-aware, hybrid voting).
4. Reference-free correction applicable in deployment.
5. Summary: paired bootstrap confidence intervals and per-scene timing
   for every protocol.
6. Error decomposition of the residual ensemble error into four
   distance-based categories — boundary-band and deep false negatives,
   boundary-band and detached false positives, in metres at each scene's
   native resolution; the dominance of the boundary band explains why gradient- and morphology-based refinement
   yields no gain on gradient-diffuse plume boundaries.

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

import os
import re
import glob
import time
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
from scipy.ndimage import binary_dilation, binary_erosion
from skimage.morphology import remove_small_objects, remove_small_holes
from skimage.segmentation import slic
from skimage.filters import sobel

# Dice-weighted averaged probability maps of the ensemble, exported by
# notebook 13 (PNG, 0-255). Binarization at > 127 corresponds to the 0.5
# threshold of the working configuration; the float maps are consumed by the
# TTA section.
INPUT_MASKS_DIR = "ensemble_test_probability/weighted_by_dice"

# Original RGB test images (required by image-guided methods)
IMAGES_DIR = "split_data/test/images"

# Ground truth of the test split (used for evaluation only)
GT_DIR = "split_data/test/Masks"

# --- Method parameters -------------------------------------------------------
OPENING_KERNEL_SIZE = 3
CLOSING_KERNEL_SIZE = 5
MIN_OBJECT_SIZE = 50

GUIDED_RADIUS = 5
GUIDED_EPS = 0.01

TTA_AUGMENTATIONS = ["original", "hflip", "vflip", "rot180", "rot90", "rot270"]
BILATERAL_D = 9
BILATERAL_SIGMA_COLOR = 75
BILATERAL_SIGMA_SPACE = 75
EDGE_SIGMA_S = 60
EDGE_SIGMA_R = 0.4

GRABCUT_ITERATIONS = 5
SLIC_N_SEGMENTS = 500
SLIC_COMPACTNESS = 10

RG_COLOR_THRESHOLD = 30
RG_SPATIAL_RADIUS = 50
RG_MIN_REGION_SIZE = 100

SMALL_OBJECT_THRESHOLD = 100
SMALL_HOLE_THRESHOLD = 100
BOUNDARY_WIDTH = 10
CONFIDENCE_KEEP_THRESHOLD = 0.75   # min component confidence kept by the confidence protocol
HONEST_COLOR_THRESHOLD = 2.0

THRESHOLD = 0.5

# --- Statistics --------------------------------------------------------------
N_BOOTSTRAP = 10000
RNG_SEED = 42

In [ ]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def match_filename(pred_name, file_dict):
    """Finds the file corresponding to a prediction name in a name->path dict."""
    base_name = pred_name.replace("_pred.png", ".png")
    if base_name in file_dict:
        return file_dict[base_name]
    m = re.search(r"(\d{4}-\d{2}-\d{2})", pred_name)
    if m:
        date = m.group(1)
        for name, path in file_dict.items():
            if date in name:
                return path
    if pred_name in file_dict:
        return file_dict[pred_name]
    return None


def dice_score(pred, gt):
    """Dice coefficient between two binary masks."""
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    if denom == 0:
        return 1.0
    return 2.0 * inter / denom


def load_binary(path, ref_shape=None):
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    if ref_shape is not None and mask.shape != ref_shape:
        mask = cv2.resize(mask, (ref_shape[1], ref_shape[0]),
                          interpolation=cv2.INTER_NEAREST)
    return (mask > 127).astype(np.uint8)


def load_gray(path, ref_shape=None):
    arr = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if arr is None:
        return None
    if ref_shape is not None and arr.shape != ref_shape:
        arr = cv2.resize(arr, (ref_shape[1], ref_shape[0]),
                         interpolation=cv2.INTER_LINEAR)
    return arr.astype(np.float32) / 255.0


def load_test_triplets():
    """Yields (name, rgb_image, probability_map, binary_mask, gt_mask)
    for every test scene."""
    gt_files = {os.path.basename(p): p
                for p in glob.glob(os.path.join(GT_DIR, "*.png"))}
    img_files = {os.path.basename(p): p
                 for p in glob.glob(os.path.join(IMAGES_DIR, "*.png"))}
    for mask_path in sorted(glob.glob(os.path.join(INPUT_MASKS_DIR, "*.png"))):
        name = os.path.basename(mask_path)
        gt_path = match_filename(name, gt_files)
        img_path = match_filename(name, img_files)
        if gt_path is None or img_path is None:
            continue
        gt = load_binary(gt_path)
        prob = load_gray(mask_path, ref_shape=gt.shape)
        pred = load_binary(mask_path, ref_shape=gt.shape)
        rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        if rgb.shape[:2] != gt.shape:
            rgb = cv2.resize(rgb, (gt.shape[1], gt.shape[0]),
                             interpolation=cv2.INTER_LINEAR)
        yield name, rgb, prob, pred, gt


def evaluate_methods(methods):
    """Applies each method to every scene and prints a summary table with the
    mean per-scene processing time of every method.

    methods: dict name -> callable(rgb, pred, prob) returning a binary mask.
    Returns dict name -> list of per-image Dice values.
    """
    scores = {"Baseline (weighted avg)": []}
    seconds = {m: 0.0 for m in methods}
    for m in methods:
        scores[m] = []
    n_scenes = 0
    for name, rgb, prob, pred, gt in tqdm(list(load_test_triplets()), desc="Scenes"):
        n_scenes += 1
        scores["Baseline (weighted avg)"].append(dice_score(pred, gt))
        for m, fn in methods.items():
            t0 = time.perf_counter()
            refined = fn(rgb, pred.copy(), prob)
            seconds[m] += time.perf_counter() - t0
            scores[m].append(dice_score(refined, gt))
    base = np.mean(scores["Baseline (weighted avg)"])
    print("\n%-38s %8s %8s %9s %9s" % ("Method", "Dice", "Std", "Delta",
                                       "s/scene"))
    print("-" * 68)
    for m, vals in scores.items():
        v = np.array(vals)
        t = seconds.get(m, 0.0) / max(n_scenes, 1)
        print("%-38s %8.4f %8.4f %+9.4f %9.2f" % (m, v.mean(), v.std(),
                                                  v.mean() - base, t))
    return scores

In [ ]:
# ============================================================================
# SECTION 1. MORPHOLOGY AND IMAGE-GUIDED SMOOTHING
# ============================================================================

def refine_morphology(rgb, pred, prob=None):
    """Opening + closing + removal of small objects and holes."""
    k_open = np.ones((OPENING_KERNEL_SIZE, OPENING_KERNEL_SIZE), np.uint8)
    k_close = np.ones((CLOSING_KERNEL_SIZE, CLOSING_KERNEL_SIZE), np.uint8)
    out = cv2.morphologyEx(pred, cv2.MORPH_OPEN, k_open)
    out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, k_close)
    out = remove_small_objects(out.astype(bool), MIN_OBJECT_SIZE)
    out = remove_small_holes(out, MIN_OBJECT_SIZE)
    return out.astype(np.uint8)


def refine_guided(rgb, pred, prob=None):
    """Guided filter on the mask with the RGB image as the guide.

    Requires opencv-contrib-python (cv2.ximgproc). Falls back to the input
    mask when the module is unavailable.
    """
    if not hasattr(cv2, "ximgproc"):
        return pred
    guide = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    soft = cv2.ximgproc.guidedFilter(guide, (pred * 255).astype(np.uint8),
                                     GUIDED_RADIUS, GUIDED_EPS * 255 * 255)
    return (soft > 127).astype(np.uint8)


def refine_combined(rgb, pred, prob=None):
    """Morphology followed by the guided filter."""
    return refine_guided(rgb, refine_morphology(rgb, pred))


scores_morph = evaluate_methods({
    "Morphology": refine_morphology,
    "Guided Filter": refine_guided,
    "Combined (morphology + guided filter)": refine_combined,
})

In [ ]:
# ============================================================================
# SECTION 2. MODEL-LEVEL TEST-TIME AUGMENTATION
# ============================================================================
# The TTA probability maps are produced by the TTA export cells of the four
# model notebooks (02, 05, 10, 12): each model is re-run on six dihedral
# transforms of the input image, the predictions are mapped back to the
# original orientation and averaged, so the maps consumed here are genuine
# re-predictions rather than transformed copies of the baseline mask. Here
# the four TTA maps are combined with the ensemble weights computed on the
# validation split (results/ensemble_weights.json, written by
# 13_ensemble.ipynb) and evaluated in three variants: plain thresholding of
# the combined TTA map, and the same map smoothed by a bilateral or an
# edge-preserving filter before thresholding.

import json

TTA_PROB_DIRS = {
    "YOLO": "probability_masks/test_tta/yolo",
    "dlv3+-ResNet50": "probability_masks/test_tta/dlv3+_with_resnet50",
    "Mask2Former": "probability_masks/test_tta/mask2former",
    "UNet-ResNet50": "probability_masks/test_tta/unet_with_resnet50",
}
WEIGHTS_JSON = "results/ensemble_weights.json"
TTA_NAME = "TTA (6 augmentations)"
TTA_BILATERAL_NAME = "TTA + Bilateral"
TTA_EDGE_NAME = "TTA + Edge-Preserving"


def smooth_bilateral(prob_map):
    """Edge-aware bilateral smoothing of a probability map."""
    as_u8 = (np.clip(prob_map, 0.0, 1.0) * 255).astype(np.uint8)
    out = cv2.bilateralFilter(as_u8, BILATERAL_D,
                              BILATERAL_SIGMA_COLOR, BILATERAL_SIGMA_SPACE)
    return out.astype(np.float32) / 255.0


def smooth_edge_preserving(prob_map):
    """Edge-preserving (recursive) smoothing of a probability map."""
    as_u8 = (np.clip(prob_map, 0.0, 1.0) * 255).astype(np.uint8)
    bgr = cv2.cvtColor(as_u8, cv2.COLOR_GRAY2BGR)
    out = cv2.edgePreservingFilter(bgr, flags=cv2.RECURS_FILTER,
                                   sigma_s=EDGE_SIGMA_S, sigma_r=EDGE_SIGMA_R)
    return cv2.cvtColor(out, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0


results_tta = {}
_ready = (all(os.path.isdir(d) for d in TTA_PROB_DIRS.values())
          and os.path.exists(WEIGHTS_JSON))
if _ready:
    with open(WEIGHTS_JSON) as f:
        _w = json.load(f)["weights"]
    _weights = np.array([_w[k] for k in TTA_PROB_DIRS], dtype=np.float64)
    _weights = _weights / _weights.sum()

    base_scores = []
    tta_scores = {TTA_NAME: [], TTA_BILATERAL_NAME: [], TTA_EDGE_NAME: []}
    tta_seconds = {TTA_NAME: 0.0, TTA_BILATERAL_NAME: 0.0, TTA_EDGE_NAME: 0.0}
    for name, rgb, prob, pred, gt in tqdm(list(load_test_triplets()), desc="TTA"):
        maps = []
        for d in TTA_PROB_DIRS.values():
            p = load_gray(os.path.join(d, name), ref_shape=gt.shape)
            if p is None:
                break
            maps.append(p)
        if len(maps) != len(TTA_PROB_DIRS):
            continue
        base_scores.append(dice_score(pred, gt))

        t0 = time.perf_counter()
        acc = np.zeros(gt.shape, dtype=np.float64)
        for w_i, p in zip(_weights, maps):
            acc += w_i * p
        tta_scores[TTA_NAME].append(
            dice_score((acc > THRESHOLD).astype(np.uint8), gt))
        tta_seconds[TTA_NAME] += time.perf_counter() - t0

        acc32 = acc.astype(np.float32)

        t0 = time.perf_counter()
        smoothed = smooth_bilateral(acc32)
        tta_scores[TTA_BILATERAL_NAME].append(
            dice_score((smoothed > THRESHOLD).astype(np.uint8), gt))
        tta_seconds[TTA_BILATERAL_NAME] += time.perf_counter() - t0

        t0 = time.perf_counter()
        smoothed = smooth_edge_preserving(acc32)
        tta_scores[TTA_EDGE_NAME].append(
            dice_score((smoothed > THRESHOLD).astype(np.uint8), gt))
        tta_seconds[TTA_EDGE_NAME] += time.perf_counter() - t0

    n_matched = len(base_scores)
    if n_matched == len(list(load_test_triplets())):
        results_tta = {"Baseline (weighted avg)": base_scores}
        results_tta.update(tta_scores)
        print("%-38s %8.4f" % ("Baseline (weighted avg)", np.mean(base_scores)))
        for key in (TTA_NAME, TTA_BILATERAL_NAME, TTA_EDGE_NAME):
            vals = tta_scores[key]
            print("%-38s %8.4f  (delta %+0.4f, %.2f s/scene)"
                  % (key, np.mean(vals),
                     np.mean(vals) - np.mean(base_scores),
                     tta_seconds[key] / max(n_matched, 1)))
    else:
        print(f"TTA exports incomplete ({n_matched} scenes matched); "
              "protocols skipped.")
else:
    print("TTA exports or results/ensemble_weights.json not found; run the "
          "TTA export cells of notebooks 02/05/10/12 and notebook 13 first.")


In [ ]:
# ============================================================================
# SECTION 3. COLOUR- AND REGION-BASED REFINEMENT
# ============================================================================

def refine_grabcut(rgb, pred, prob=None):
    """GrabCut initialised from the ensemble mask."""
    if pred.sum() == 0:
        return pred
    gc_mask = np.full(pred.shape, cv2.GC_PR_BGD, np.uint8)
    gc_mask[pred == 1] = cv2.GC_PR_FGD
    sure_fg = binary_erosion(pred, iterations=5)
    sure_bg = binary_erosion(1 - pred, iterations=5)
    gc_mask[sure_fg] = cv2.GC_FGD
    gc_mask[sure_bg.astype(bool)] = cv2.GC_BGD
    bgd, fgd = np.zeros((1, 65), np.float64), np.zeros((1, 65), np.float64)
    try:
        cv2.grabCut(rgb, gc_mask, None, bgd, fgd,
                    GRABCUT_ITERATIONS, cv2.GC_INIT_WITH_MASK)
    except cv2.error:
        return pred
    return np.isin(gc_mask, (cv2.GC_FGD, cv2.GC_PR_FGD)).astype(np.uint8)


def refine_superpixel(rgb, pred, prob=None):
    """SLIC superpixels; a segment is kept when the mean mask value exceeds 0.5."""
    segments = slic(rgb, n_segments=SLIC_N_SEGMENTS,
                    compactness=SLIC_COMPACTNESS, start_label=0)
    out = np.zeros_like(pred)
    for seg_id in np.unique(segments):
        sel = segments == seg_id
        if pred[sel].mean() > 0.5:
            out[sel] = 1
    return out


def refine_region_growing(rgb, pred, prob=None):
    """Constrained dilation towards colour-similar pixels in LAB space."""
    if pred.sum() == 0:
        return pred
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    obj_mean = lab[pred.astype(bool)].mean(axis=0)
    similar = (np.linalg.norm(lab - obj_mean, axis=-1)
               < RG_COLOR_THRESHOLD).astype(np.uint8)
    grown = pred.copy()
    for _ in range(RG_SPATIAL_RADIUS // 5):
        grown = binary_dilation(grown, iterations=5) & similar.astype(bool)
        grown = grown.astype(np.uint8)
    out = remove_small_objects(grown.astype(bool), RG_MIN_REGION_SIZE)
    return (out | pred.astype(bool)).astype(np.uint8)


def refine_adaptive_color(rgb, pred, prob=None):
    """Adds boundary pixels whose LAB colour lies within 2 std of the object."""
    if pred.sum() == 0:
        return pred
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    obj = lab[pred.astype(bool)]
    mean, std = obj.mean(axis=0), obj.std(axis=0) + 1e-6
    band = binary_dilation(pred, iterations=BOUNDARY_WIDTH) & ~pred.astype(bool)
    z = np.abs((lab - mean) / std).max(axis=-1)
    add = band & (z < 2.0)
    return (pred.astype(bool) | add).astype(np.uint8)


def refine_texture(rgb, pred, prob=None):
    """Removes boundary pixels with a high local gradient (simplified
    texture-aware variant based on the Sobel magnitude)."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    grad = sobel(gray)
    thr = np.percentile(grad, 90)
    band = pred.astype(bool) & ~binary_erosion(pred, iterations=BOUNDARY_WIDTH)
    out = pred.copy()
    out[band & (grad > thr)] = 0
    return out


def refine_hybrid(rgb, pred, prob=None):
    """Majority vote of GrabCut, region growing and adaptive colour."""
    votes = (refine_grabcut(rgb, pred).astype(np.int32)
             + refine_region_growing(rgb, pred)
             + refine_adaptive_color(rgb, pred))
    return (votes >= 2).astype(np.uint8)


scores_color = evaluate_methods({
    "GrabCut": refine_grabcut,
    "Superpixel (SLIC)": refine_superpixel,
    "Region Growing": refine_region_growing,
    "Adaptive Color": refine_adaptive_color,
    "Texture-Aware": refine_texture,
    "Hybrid (voting)": refine_hybrid,
})

In [ ]:
# ============================================================================
# SECTION 4. GT-FREE REFINEMENT SET
# ============================================================================
# All methods in this section operate without access to the reference masks
# and are therefore applicable in deployment.

def honest_morphology(rgb, pred, prob=None):
    out = remove_small_objects(pred.astype(bool), SMALL_OBJECT_THRESHOLD)
    out = remove_small_holes(out, SMALL_HOLE_THRESHOLD)
    return out.astype(np.uint8)


def honest_color_boundary(rgb, pred, prob=None):
    """Snaps the boundary band to object-like colours (LAB z-score test)."""
    if pred.sum() == 0:
        return pred
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    obj = lab[binary_erosion(pred, iterations=BOUNDARY_WIDTH)]
    if len(obj) == 0:
        obj = lab[pred.astype(bool)]
    mean, std = obj.mean(axis=0), obj.std(axis=0) + 1e-6
    z = np.abs((lab - mean) / std).max(axis=-1)
    inner = binary_erosion(pred, iterations=BOUNDARY_WIDTH)
    band_out = binary_dilation(pred, iterations=BOUNDARY_WIDTH) & ~pred.astype(bool)
    band_in = pred.astype(bool) & ~inner
    out = inner.copy()
    out |= band_in & (z < HONEST_COLOR_THRESHOLD)
    out |= band_out & (z < HONEST_COLOR_THRESHOLD)
    return out.astype(np.uint8)


def honest_confidence(rgb, pred, prob=None):
    """Removes low-confidence detections: connected components of the binary
    mask whose maximum ensemble probability stays below
    CONFIDENCE_KEEP_THRESHOLD are discarded; the rest of the mask is kept
    unchanged."""
    if prob is None or pred.sum() == 0:
        return pred
    from skimage.measure import label as _cc_label
    lab = _cc_label(pred)
    out = np.zeros_like(pred)
    for region_id in range(1, lab.max() + 1):
        sel = lab == region_id
        if prob[sel].max() >= CONFIDENCE_KEEP_THRESHOLD:
            out[sel] = 1
    return out


def honest_hybrid(rgb, pred, prob=None):
    return honest_color_boundary(rgb, honest_morphology(rgb, pred))


scores_honest = evaluate_methods({
    "Morphology only": honest_morphology,
    "Color Boundary": honest_color_boundary,
    "Confidence-based": honest_confidence,
    "Hybrid (morphology + color boundary)": honest_hybrid,
})

In [ ]:
# ============================================================================
# SECTION 5. SUMMARY: PAIRED BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================================
# 95% bootstrap confidence intervals (N_BOOTSTRAP resamples of the test
# scenes) for the mean Dice of every protocol and for its paired difference
# with the baseline. All sections iterate the test scenes in the same sorted
# order, so the per-scene lists are aligned.

all_scores = {}
for d in (scores_morph, results_tta, scores_color, scores_honest):
    for k, v in d.items():
        if k == "Baseline (weighted avg)" and k in all_scores:
            continue
        all_scores[k] = np.asarray(v, dtype=np.float64)

_lengths = {len(v) for v in all_scores.values()}
assert len(_lengths) == 1, f"Scene lists are not aligned: {_lengths}"
n_scenes = _lengths.pop()

rng = np.random.default_rng(RNG_SEED)
baseline = all_scores["Baseline (weighted avg)"]


def _boot_ci(values):
    means = np.empty(N_BOOTSTRAP)
    for b in range(N_BOOTSTRAP):
        take = rng.integers(0, n_scenes, n_scenes)
        means[b] = values[take].mean()
    return np.percentile(means, [2.5, 97.5])


header = f"{'Protocol':<38} {'Dice [95% CI]':<24} Delta vs baseline [95% CI]"
print(header)
print("-" * len(header))
for name, vals in all_scores.items():
    lo, hi = _boot_ci(vals)
    if name == "Baseline (weighted avg)":
        print(f"{name:<38} {vals.mean():.3f} [{lo:.3f}, {hi:.3f}]")
        continue
    diff = vals - baseline
    dlo, dhi = _boot_ci(diff)
    print(f"{name:<38} {vals.mean():.3f} [{lo:.3f}, {hi:.3f}]   "
          f"{diff.mean():+.4f} [{dlo:+.4f}, {dhi:+.4f}]")

In [ ]:
# ============================================================================
# SECTION 6. ERROR DECOMPOSITION OF THE ENSEMBLE MASKS
# ============================================================================
# Every mispredicted pixel is assigned to one of four mutually exclusive
# categories by its Euclidean distance to the reference plume boundary,
# measured in metres at the native resolution of the scene:
#   FN band / FP band       - errors within +-W metres of the boundary
#   FN deep / FP detached   - errors farther than W metres from it
# The four categories cover 100% of the residual error by construction.
# Areas are aggregated in square metres over all scenes, so Sentinel-2 (10 m)
# and Landsat (30 m) scenes contribute on a common physical scale. The band
# share is reported for several half-widths as a sensitivity check.

from scipy.ndimage import distance_transform_edt

GSD_KEYWORDS = {
    10.0: ("s2", "sentinel", "msi"),
    30.0: ("landsat", "l8", "l9", "lc8", "lc08", "lc09", "oli"),
}
DEFAULT_GSD_M = 10.0

def gsd_of(name):
    low = name.lower()
    for gsd, keys in GSD_KEYWORDS.items():
        if any(k in low for k in keys):
            return gsd
    return DEFAULT_GSD_M

BOUNDARY_BANDS_M = (50.0, 100.0, 200.0)   # band half-widths; 100 m is the headline

totals = {w: {"FN band": 0.0, "FP band": 0.0, "FN deep": 0.0, "FP detached": 0.0}
          for w in BOUNDARY_BANDS_M}
unknown_gsd = 0

for name, rgb, prob, pred, gt in tqdm(list(load_test_triplets()), desc="Errors"):
    if gt.sum() == 0:
        continue
    low = name.lower()
    if not any(k in low for keys in GSD_KEYWORDS.values() for k in keys):
        unknown_gsd += 1
    gsd = gsd_of(name)
    px_area = gsd * gsd                      # m^2 per pixel

    fp = (pred == 1) & (gt == 0)
    fn = (pred == 0) & (gt == 1)

    # Distance of every pixel to the reference boundary, in metres: inside the
    # mask - distance to the nearest background pixel, outside - distance to
    # the nearest mask pixel.
    dist_in = distance_transform_edt(gt) * gsd
    dist_out = distance_transform_edt(1 - gt) * gsd

    for w in BOUNDARY_BANDS_M:
        totals[w]["FN band"] += px_area * float((fn & (dist_in <= w)).sum())
        totals[w]["FN deep"] += px_area * float((fn & (dist_in > w)).sum())
        totals[w]["FP band"] += px_area * float((fp & (dist_out <= w)).sum())
        totals[w]["FP detached"] += px_area * float((fp & (dist_out > w)).sum())

if unknown_gsd:
    print(f"[warn] sensor not recognised from {unknown_gsd} file names; "
          f"assumed {DEFAULT_GSD_M:.0f} m GSD - extend GSD_KEYWORDS if needed")

for w in BOUNDARY_BANDS_M:
    t = totals[w]
    total_err = sum(t.values())
    band = t["FN band"] + t["FP band"]
    print(f"\nBoundary band +-{w:.0f} m  "
          f"(total residual error {total_err / 1e6:.2f} km^2):")
    for k in ("FN band", "FP band", "FN deep", "FP detached"):
        print(f"  {k:12s} {100.0 * t[k] / total_err:6.2f} % of the residual error")
    print(f"  boundary-band share (FN band + FP band): {100.0 * band / total_err:.2f} %")

Methods requiring the reference masks for correction are intentionally
excluded: they are inapplicable in deployment. The reference masks are used
in this notebook for measurement only.